### Marktwaage – Lösung Schritt für Schritt

*40. Bundeswettbewerb Informatik (2021), 1. Runde, Aufgabe 5*

Auf dem Dachboden durchstöbert Marie eine Kiste mit alten Gegenständen ihrer Urgroßmutter. Dabei findet sie eine hübsche
Marktwaage mit dazugehörigen Gewichtsstücken: 5 kg, 1 kg, 1 kg, 1 kg, 500 g, 500 g, 500 g, 100 g, 100 g, 100 g, 50 g, 50 g,
10 g, 10 g und nochmals 10 g.

Marie sagt erstaunt zu ihrer Mutter: „Damit kann man doch nur wenige verschiedene Gewichte zusammenstellen. Es ist ja gar
nicht möglich, z. B. 440 Gramm Kartoffeln abzuwiegen.“

Ihre Mutter erwidert: „Doch, hierzu stellt man ein 500-Gramm-Stück auf eine Seite der Waage und die Kartoffeln mit einem
10-Gramm- und einem 50-Gramm-Stück auf die andere Seite.“

Damit hatte Marie nicht gerechnet. Jetzt fragt sie sich, wie groß die Menge der möglichen Gewichte von Waren ist, die man
mit dieser Waage wiegen kann. Die Menge enthält offenbar 440 g; aber sollte sie womöglich alle Vielfachen von 10 g zwischen
10 g und 10 kg enthalten?

Nachdem sie eine Weile über diese Frage nachgedacht hat, grübelt sie noch darüber nach, wie sich die Situation wohl
darstellt, wenn zu solch einer Waage ein anderer und vielleicht „krummer“ Satz von Gewichtsstücken vorhanden ist, etwa
mit fünf Stücken zu je 2 kg, sechs zu je 371 g, drei zu je 127 g und schließlich drei Stücken zu je 42 g.

#### Aufgabe 5

Schreibe ein Programm, das solche Fragestellungen beantwortet. Dein Programm soll eine Liste von Gewichten der
Gewichtsstücke einlesen und in 10g-Schritten alle Gewichte zwischen 10g und 10kg untersuchen. Für jedes untersuchte Gewicht
soll das Programm ausgeben, ob sich das Gewicht mit den vorhandenen Gewichtsstücken erzielen lässt. Falls ja, soll außerdem
ausgegeben werden, welche Gewichtsstücke dabei auf welche Seite der Waage gestellt werden müssen, wenn die zu wiegende Ware
immer auf die linke Seite gelegt wird. Falls nein, soll ausgegeben werden, wie nahe man an das Zielgewicht man mit den
vorhandenen Gewichtsstücken kommen kann.

Wende dein Programm mindestens auf alle Beispiele an, die du auf den BWINF-Webseiten findest, und dokumentiere die Ergebnisse.

#### Beispieldaten (`beispieldaten/gewichtsstuecke0.txt` bis `gewichtsstuecke5.txt`)

- 1. Zeile: Anzahl der verschiedenen Gewichte
- danach je Zeile ein Gewicht in Gramm und die Anzahl der Stücke mit diesem Gewicht

## Was dir bei dieser Aufgabe hilft

**Algorithmische Kenntnisse**
- **Alle Möglichkeiten aufzählen (Backtracking, Durchprobieren):** Jedes Gewichtsstück liegt links, rechts oder gar nicht auf der Waage. Man
  probiert alle Verteilungen durch und merkt sich jeweils die Summe.
- **Die Aufgabe umformulieren:** Die Ware liegt links, dazu vielleicht Gewichtsstücke. Rechts liegen andere Gewichtsstücke. Im Gleichgewicht gilt
  *Ware + Gewichte links = Gewichte rechts*, also *Ware = Gewichte rechts − Gewichte links*. Ein Gewichtsstück zählt also **plus**, wenn es rechts liegt,
  **minus**, wenn es links liegt, und **0**, wenn es nicht benutzt wird. Für jedes Gewicht `w`, das es `n`-mal gibt, wählt man einen **Faktor** von `-n` bis `n`.
- **Sortierte Liste und Binärsuche:** Hat man alle erreichbaren Summen sortiert, findet man den Wert, der einem Zielgewicht am nächsten liegt,
  ohne die ganze Liste durchzugehen.
- **Teile und kombiniere (Meet in the Middle):** Sind es zu viele Verteilungen für eine Liste, teilt man die Gewichte in zwei Hälften, bildet für jede
  Hälfte die Summen und setzt beim Suchen zwei Teilsummen zusammen. Statt `K` Verteilungen muss man nur etwa `2·√K` Summen speichern.

**Nützliche Funktionen und Bibliotheken**
- `dict` (Wörterbuch): `summen[s]` merkt sich zu jeder Summe `s` eine Verteilung; `s in summen` ist sehr schnell.
- `sorted(...)` und `bisect.bisect_left(liste, x)`: Binärsuche in einer sortierten Liste. Liefert die Stelle, an der `x` einsortiert würde.
- `range(-n, n + 1)` für die Faktoren von `-n` bis `n`.

## Die Idee

**Faktoren.** Für jedes Gewicht `w` mit `n` Stücken gibt es einen Faktor `f` zwischen `-n` und `n`: Bei `f = 2` liegen zwei Stücke rechts, bei `f = -1` liegt ein
Stück links (bei der Ware), bei `f = 0` keines. Die Ware wiegt dann `Summe der w · f`.

Beispiel der Lösungshinweise: Gewichte 5 g, 5 g, 20 g. Dann ist der Faktor für die 5 g einer von `-2, -1, 0, 1, 2` und für die 20 g einer von `-1, 0, 1`.
Das ergibt `5 · 3 = 15` Kombinationen. Allgemein ist die Zahl der Kombinationen `(2·n₁ + 1) · (2·n₂ + 1) · …`.

**Spiegelung.** Zu jeder Verteilung mit der Summe `s` gibt es die vertauschte Verteilung mit `-s`. Wir brauchen nur die Summen `>= 10`.

**Mehrere Verteilungen mit gleicher Summe.** Dasselbe Gewicht lässt sich oft auf mehrere Arten wiegen. Wir nehmen die Verteilung mit den **wenigsten Gewichtsstücken**
(wie in den Lösungshinweisen).

**Schritt für Schritt aufbauen.** Wir nehmen die Gewichte nacheinander dazu. Wir merken uns für jede bisher erreichbare Summe *eine* beste Verteilung. Kommt ein neues Gewicht dazu,
bildet jede alte Summe mit jedem Faktor eine neue. Verschiedene Wege zur gleichen Summe werden dabei in einem Wörterbuch zusammengefasst, das spart Platz.

## Schritt 1: Einlesen

In [ ]:
def lies_gewichte(dateiname):
    with open(dateiname, encoding='utf-8-sig') as f:
        zeilen = [zeile.split() for zeile in f.read().splitlines() if zeile.strip()]
    gewichte = []
    for zeile in zeilen[1:]:                       # die erste Zeile ist die Zahl der Gewichte
        gewichte.append((int(zeile[0]), int(zeile[1])))   # (Gewicht, Anzahl)
    return gewichte


gew0 = lies_gewichte('beispieldaten/gewichtsstuecke0.txt')
print(gew0)

[(10, 3), (50, 2), (100, 3), (500, 3), (1000, 3), (5000, 1)]


## Schritt 2: Alle erreichbaren Summen

`alle_summen(gewichte)` liefert ein Wörterbuch. Der Schlüssel ist eine Summe, der Wert ist ein Paar `(Zahl der Stücke, Faktoren)`. Die Faktoren stehen in der Reihenfolge der Gewichte.

- Start: Ohne Gewichte ist nur die Summe `0` erreichbar, mit 0 Stücken und leerer Faktorenliste.
- Für jedes Gewicht `(w, n)` bauen wir ein neues Wörterbuch `neu`. Jede alte Summe `s` wird mit jedem Faktor `f` von `-n` bis `n` zu `s + f·w`.
  Benutzt wurden dabei `abs(f)` Stücke mehr.
- Gibt es die neue Summe schon, behalten wir die Verteilung mit weniger Stücken.

In [ ]:
def alle_summen(gewichte):
    summen = {0: (0, ())}
    for w, n in gewichte:
        neu = {}
        for s in summen:
            stuecke, faktoren = summen[s]
            for f in range(-n, n + 1):
                s2 = s + f * w
                st2 = stuecke + abs(f)
                if s2 not in neu or st2 < neu[s2][0]:
                    neu[s2] = (st2, faktoren + (f,))
        summen = neu
    return summen

Probe mit dem Beispiel 5 g, 5 g, 20 g aus den Lösungshinweisen. Nach der Spiegelung bleiben die Summen von 0 bis 30:

In [ ]:
klein = alle_summen([(5, 2), (20, 1)])
for s in sorted(klein):
    if s >= 0:
        print(s, klein[s])

0 (0, (0, 0))
5 (1, (1, 0))
10 (2, (2, 0))
15 (2, (-1, 1))
20 (1, (0, 1))
25 (2, (1, 1))
30 (3, (2, 1))


Die 10 g gibt es zweimal: als `2 · 5 g` (zwei Stücke) und als `-2 · 5 g + 20 g` (drei Stücke). Das Wörterbuch behält die Verteilung mit zwei Stücken.

## Schritt 3: Das beste Ergebnis zu einem Zielgewicht

`finde(ziel, summen, schluessel)` sucht die Summe, die `ziel` am nächsten liegt.

- Ist `ziel in summen`, gibt es eine **genaue** Lösung.
- Sonst suchen wir in der sortierten Liste `schluessel` aller Summen. `bisect_left(schluessel, ziel)` liefert die Stelle `i`, an der `ziel` einsortiert würde. Die beiden Nachbarn
  `schluessel[i-1]` (kleiner) und `schluessel[i]` (größer) sind die Kandidaten. Wir nehmen den näheren, bei Gleichstand den kleineren.

Das Ergebnis ist `(erreicht, stuecke, faktoren)`.

In [ ]:
from bisect import bisect_left


def finde(ziel, summen, schluessel):
    if ziel in summen:
        return ziel, summen[ziel][0], summen[ziel][1]
    i = bisect_left(schluessel, ziel)
    kandidaten = []
    if i > 0:
        kandidaten.append(schluessel[i - 1])
    if i < len(schluessel):
        kandidaten.append(schluessel[i])
    beste = kandidaten[0]
    for k in kandidaten:
        if abs(k - ziel) < abs(beste - ziel):
            beste = k
    return beste, summen[beste][0], summen[beste][1]

## Schritt 4: Verständliche Ausgabe

Aus den Faktoren wird ein Satz: Ein negativer Faktor heißt „liegt links bei der Ware“, ein positiver „liegt rechts“.

In [ ]:
def seite(gewichte, faktoren, vorzeichen):
    teile = []
    for (w, n), f in zip(gewichte, faktoren):
        if f * vorzeichen > 0:
            teile.append(str(abs(f)) + '×' + str(w) + ' g')
    if teile:
        return ' + '.join(teile)
    return '-'


def beschreibe(ziel, gewichte, ergebnis):
    erreicht, stuecke, faktoren = ergebnis
    text = str(ziel) + ' g: '
    if erreicht == ziel:
        text += 'genau'
    else:
        text += 'nur ' + str(erreicht) + ' g (Abweichung ' + str(abs(erreicht - ziel)) + ' g)'
    text += ', links ' + seite(gewichte, faktoren, -1) + ' | rechts ' + seite(gewichte, faktoren, 1)
    return text

## Schritt 5: Das Marie-Beispiel

Marie hat 5 kg, 3 × 1 kg, 3 × 500 g, 3 × 100 g, 2 × 50 g und 3 × 10 g. Das ist genau `gewichtsstuecke0.txt`. Wir schauen uns die 440 g an
und einige weitere Gewichte:

In [ ]:
summen0 = alle_summen(gew0)
schluessel0 = sorted(summen0)
print('Verschiedene Summen:', len(summen0))
for ziel in [10, 440, 1000, 4990, 5010, 9990, 10000]:
    print(beschreibe(ziel, gew0, finde(ziel, summen0, schluessel0)))

Verschiedene Summen: 1987
10 g: genau, links - | rechts 1×10 g
440 g: genau, links 1×10 g + 1×50 g | rechts 1×500 g
1000 g: genau, links - | rechts 1×1000 g
4990 g: genau, links 1×10 g | rechts 1×5000 g
5010 g: genau, links - | rechts 1×10 g + 1×5000 g
9990 g: nur 9930 g (Abweichung 60 g), links - | rechts 3×10 g + 2×50 g + 3×100 g + 3×500 g + 3×1000 g + 1×5000 g
10000 g: nur 9930 g (Abweichung 70 g), links - | rechts 3×10 g + 2×50 g + 3×100 g + 3×500 g + 3×1000 g + 1×5000 g


Die 440 g wiegt man mit 500 g rechts und 10 g + 50 g bei der Ware, wie Maries Mutter sagt. Das sind drei Stücke, weniger geht nicht.

## Schritt 6: Alle Zielgewichte einer Datei untersuchen

`untersuche(dateiname)` geht alle Zielgewichte von 10 g bis 10 000 g in Schritten von 10 g durch und zählt die genauen Treffer. Dazu gibt es die Zahl der Kombinationen
`(2·n₁ + 1) · (2·n₂ + 1) · …`, die ein Durchprobieren aller Verteilungen bräuchte.

In [ ]:
def zahl_kombinationen(gewichte):
    k = 1
    for w, n in gewichte:
        k = k * (2 * n + 1)
    return k


def untersuche(dateiname):
    gewichte = lies_gewichte(dateiname)
    summen = alle_summen(gewichte)
    schluessel = sorted(summen)
    ergebnisse = []
    for ziel in range(10, 10001, 10):
        ergebnisse.append((ziel, finde(ziel, summen, schluessel)))
    return gewichte, ergebnisse


def bericht(gewichte, ergebnisse):
    genau = 0
    schlechteste = 0
    for ziel, (erreicht, stuecke, faktoren) in ergebnisse:
        if erreicht == ziel:
            genau = genau + 1
        else:
            schlechteste = max(schlechteste, abs(erreicht - ziel))
    print('Kombinationen:', zahl_kombinationen(gewichte), '| genau erreichbar:', genau, 'von', len(ergebnisse),
          '| größte Abweichung:', schlechteste, 'g')
    return genau

In [ ]:
ERG = {}                                   # Ergebnisse aller Dateien für den Vergleich weiter unten
for nr in range(5):
    dateiname = 'beispieldaten/gewichtsstuecke' + str(nr) + '.txt'
    gew, erg = untersuche(dateiname)
    print('gewichtsstuecke' + str(nr) + '.txt:', end=' ')
    ERG[nr] = (gew, erg, bericht(gew, erg))

gewichtsstuecke0.txt: Kombinationen: 36015 | genau erreichbar: 993 von 1000 | größte Abweichung: 70 g
gewichtsstuecke1.txt: Kombinationen: 7007 | genau erreichbar: 300 von 1000 | größte Abweichung: 4 g
gewichtsstuecke2.txt: Kombinationen: 59049 | genau erreichbar: 1000 von 1000 | größte Abweichung: 0 g
gewichtsstuecke3.txt: Kombinationen: 2187 | genau erreichbar: 1000 von 1000 | größte Abweichung: 0 g
gewichtsstuecke4.txt: Kombinationen: 27783 | genau erreichbar: 610 von 1000 | größte Abweichung: 396 g


Diese Zahlen stimmen mit den Tabellen der Lösungshinweise überein (dort „Exakte Lösungen“ und „Anzahl der Kombinationen“). Die Datei 5 fehlt noch (siehe unten).

Zum Ansehen drucken wir bei der Datei `gewichtsstuecke1.txt` (die „krummen“ Gewichte 42, 127, 371 und 2000 g) einige Zeilen aus.

In [ ]:
gew1, erg1, genau1 = ERG[1]
for ziel, ergebnis in erg1[:6]:
    print(beschreibe(ziel, gew1, ergebnis))
print('...')
for ziel, ergebnis in erg1:
    if ziel in (440, 1000, 5000, 9990):
        print(beschreibe(ziel, gew1, ergebnis))

10 g: genau, links 1×371 g | rechts 3×127 g
20 g: nur 19 g (Abweichung 1 g), links 3×42 g + 5×371 g | rechts 1×2000 g
30 g: nur 29 g (Abweichung 1 g), links 3×42 g + 6×371 g | rechts 3×127 g + 1×2000 g
40 g: nur 42 g (Abweichung 2 g), links - | rechts 1×42 g
50 g: nur 51 g (Abweichung 1 g), links 2×42 g + 3×127 g + 4×371 g | rechts 1×2000 g
60 g: genau, links 1×127 g + 5×371 g | rechts 1×42 g + 1×2000 g
...
440 g: nur 441 g (Abweichung 1 g), links 5×371 g | rechts 1×42 g + 2×127 g + 1×2000 g
1000 g: nur 997 g (Abweichung 3 g), links 3×42 g | rechts 3×127 g + 2×371 g
5000 g: nur 4997 g (Abweichung 3 g), links 3×42 g | rechts 3×127 g + 2×371 g + 2×2000 g
9990 g: genau, links 3×127 g | rechts 1×371 g + 5×2000 g


Bei Datei 2 (Gewichte 10, 20, 40, … g, je einmal) ist jedes Vielfache von 10 g erreichbar, bei Datei 3 (10, 30, 90, … g, je einmal) ebenfalls. Das kennt man vom Binärsystem
und vom Dreiersystem mit den Ziffern −1, 0, 1 (ausgeglichenes Ternärsystem).

In [ ]:
for nr in (2, 3):
    gew, erg, genau = ERG[nr]
    print('Datei', nr, ': genau erreichbar', genau, 'von', len(erg))
    for ziel, ergebnis in erg:
        if ziel in (10, 100, 440, 10000):
            print('  ', beschreibe(ziel, gew, ergebnis))

Datei 2 : genau erreichbar 1000 von 1000
   10 g: genau, links - | rechts 1×10 g
   100 g: genau, links - | rechts 1×20 g + 1×80 g
   440 g: genau, links 1×40 g + 1×160 g | rechts 1×640 g
   10000 g: genau, links - | rechts 1×80 g + 1×320 g + 1×640 g + 1×1280 g + 1×2560 g + 1×5120 g
Datei 3 : genau erreichbar 1000 von 1000
   10 g: genau, links - | rechts 1×10 g
   100 g: genau, links - | rechts 1×10 g + 1×90 g
   440 g: genau, links 1×10 g + 1×90 g + 1×270 g | rechts 1×810 g
   10000 g: genau, links - | rechts 1×10 g + 1×270 g + 1×2430 g + 1×7290 g


## Schritt 7: Für Neugierige: Datei 5 mit Meet in the Middle

Die Datei `gewichtsstuecke5.txt` hat sehr große Gewichte (bis über 1,9 Milliarden Gramm) und deutlich mehr Gewichte. Die Zahl der Kombinationen liegt über 100 Millionen. Für alle würde
unser Wörterbuch viel zu viel Platz und Zeit brauchen. Das zeigt schon die Rechnung:

In [ ]:
gew5 = lies_gewichte('beispieldaten/gewichtsstuecke5.txt')
print('Kombinationen:', zahl_kombinationen(gew5))

Kombinationen: 101269035


**Idee:** Wir teilen die Gewichte in eine **linke** und eine **rechte Hälfte** (hier: die vorderen Gewichte und die hinteren, als Gruppen der Gewichtsliste, nicht Waagenseiten). Jede
Hälfte hat nur rund zehntausend Summen. Eine Verteilung für die Ware ist dann `summe_vorne + summe_hinten = ziel`. Für jede `summe_vorne` suchen wir im Wörterbuch der hinteren Hälfte
nach `ziel - summe_vorne`. Gibt es keine genaue Lösung, suchen wir mit `bisect_left` die nächste hintere Summe.

Die Hälften werden so gewählt, dass die Zahl der Kombinationen etwa gleich groß ist:

In [ ]:
def teile_gewichte(gewichte):
    gesamt = zahl_kombinationen(gewichte)
    beste_stelle = 0
    kleinster_unterschied = None
    for stelle in range(len(gewichte) + 1):
        vorn = zahl_kombinationen(gewichte[:stelle])
        hinten = gesamt // vorn
        unterschied = abs(vorn - hinten)
        if kleinster_unterschied is None or unterschied < kleinster_unterschied:
            kleinster_unterschied = unterschied
            beste_stelle = stelle
    return gewichte[:beste_stelle], gewichte[beste_stelle:]


vorn5, hinten5 = teile_gewichte(gew5)
sv5 = alle_summen(vorn5)
sh5 = alle_summen(hinten5)
sh5_sortiert = sorted(sh5)
print('Gewichte vorn:', len(vorn5), '(', len(sv5), 'Summen ), hinten:', len(hinten5), '(', len(sh5), 'Summen )')

Gewichte vorn: 7 ( 8505 Summen ), hinten: 6 ( 11907 Summen )


Jetzt die Suche mit zwei Hälften. Bei den **genauen** Treffern wählen wir unter allen Möglichkeiten die mit den wenigsten Stücken. Gibt es keinen genauen Treffer, nehmen wir für jede
vordere Summe den nächsten hinteren Nachbarn und behalten die beste Abweichung (bei Gleichstand die kleinere Summe).

In [ ]:
def finde_zwei(ziel, sv, sh, sh_sortiert):
    beste = None                                     # (Stücke, Summe, Faktoren)
    for a in sv:
        b = ziel - a
        if b in sh:
            stuecke = sv[a][0] + sh[b][0]
            if beste is None or stuecke < beste[0]:
                beste = (stuecke, ziel, sv[a][1] + sh[b][1])
    if beste is not None:
        return ziel, beste[0], beste[2]
    naechste = None                                  # (Abweichung, Summe, Stücke, Faktoren)
    for a in sv:
        i = bisect_left(sh_sortiert, ziel - a)
        for j in (i - 1, i):
            if 0 <= j < len(sh_sortiert):
                b = sh_sortiert[j]
                summe = a + b
                kandidat = (abs(summe - ziel), summe, sv[a][0] + sh[b][0], sv[a][1] + sh[b][1])
                if naechste is None or kandidat[:3] < naechste[:3]:
                    naechste = kandidat
    return naechste[1], naechste[2], naechste[3]

Zur Probe lassen wir die Zwei-Hälften-Suche auch bei der Datei `gewichtsstuecke4.txt` laufen und vergleichen mit der einfachen Suche. Die Zahl der Stücke und die Abweichung müssen übereinstimmen,
die Verteilung selbst kann sich bei gleich guten Möglichkeiten unterscheiden.

In [ ]:
gew4, erg4, genau4 = ERG[4]
vorn4, hinten4 = teile_gewichte(gew4)
sv4, sh4 = alle_summen(vorn4), alle_summen(hinten4)
sh4_sortiert = sorted(sh4)
gleich = 0
for ziel, (erreicht, stuecke, faktoren) in erg4:
    e2, s2, f2 = finde_zwei(ziel, sv4, sh4, sh4_sortiert)
    if abs(e2 - ziel) == abs(erreicht - ziel) and (erreicht != ziel or s2 == stuecke):
        gleich = gleich + 1
print('Gleich gute Ergebnisse bei Datei 4:', gleich, 'von', len(erg4))

Gleich gute Ergebnisse bei Datei 4: 1000 von 1000


Nun alle 1000 Zielgewichte der Datei 5:

In [ ]:
erg5 = []
for ziel in range(10, 10001, 10):
    erg5.append((ziel, finde_zwei(ziel, sv5, sh5, sh5_sortiert)))
genau5 = sum(1 for ziel, (erreicht, st, fa) in erg5 if erreicht == ziel)
groesste5 = max(abs(erreicht - ziel) for ziel, (erreicht, st, fa) in erg5)
print('Kombinationen:', zahl_kombinationen(gew5), '| genau erreichbar:', genau5, 'von', len(erg5), '| größte Abweichung:', groesste5, 'g')
for ziel, ergebnis in erg5:
    if ziel in (10, 110, 5000):
        print(beschreibe(ziel, gew5, ergebnis))

Kombinationen: 101269035 | genau erreichbar: 84 von 1000 | größte Abweichung: 255 g
10 g: nur 11 g (Abweichung 1 g), links - | rechts 1×11 g
110 g: nur 109 g (Abweichung 1 g), links 1×11 g + 1×99480 g + 2×299836 g | rechts 1×99511 g + 1×599761 g
5000 g: genau, links 2×299836 g + 2×4497786 g + 1×1499171 g | rechts 1×599761 g + 1×10499654 g


Die Datei 5 ist mit ihren riesigen Gewichten ein Extremfall: Fast alle Zielgewichte kann man nur ungefähr treffen. Viele Gewichte liegen nämlich weit über 10 kg und tragen deshalb
nur dann etwas bei, wenn sich ihre Differenzen fast aufheben. Nur 84 der 1000 Zielgewichte sind genau erreichbar, so wie auch die Lösungshinweise angeben.

## Zusammenfassung

- Jedes Gewichtsstück zählt plus (rechts), minus (links bei der Ware) oder null. Mit Faktoren von `-n` bis `n` für `n` gleiche Stücke erhält man alle Verteilungen.
- Ein Wörterbuch fasst gleiche Summen zusammen und merkt sich die Verteilung mit den wenigsten Stücken.
- Das nächstliegende Gewicht findet man mit einer Binärsuche (`bisect_left`) in der sortierten Summenliste.
- Bei sehr vielen Kombinationen (Datei 5) teilt man die Gewichte in zwei Hälften und setzt Teilsummen zusammen („Meet in the Middle“).